<a href="https://colab.research.google.com/github/sangramshitole07/PNLP/blob/main/PNL_Lab_MSE_22_Sangram_Shitole.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Task A — Record Extraction and Search (CO1)
Using Q22_Corpus_CO1.csv, create a working application that reads the records, identifies their
UNI/26/NNN reference codes and retrieves the original messages using a text-based index. The
application must support two-term AND and AND NOT queries. Demonstrate the searches
“project AND submitted” and “project AND NOT submitted”, together with a mixed-case query
and a search containing a term absent from the collection. Show the matching record identifiers
and original messages. Explain one relevant message that a strict word-based search might fail to
retrieve

In [17]:

import pandas as pd
import re
from collections import defaultdict


df = pd.read_csv("Q22_Corpus_CO1.csv")


def extract_reference(message):
    match = re.search(
        r'UNI/\d{2}/\d{3}',
        message,
        re.IGNORECASE
    )
    return match.group(0) if match else None

df["reference"] = df["message"].apply(extract_reference)


inverted_index = defaultdict(set)

for _, row in df.iterrows():
    record_id = row["record_id"]
    words = re.findall(r'\b\w+\b', row["message"].lower())

    for word in words:
        inverted_index[word].add(record_id)


def search_and(term1, term2):
    return inverted_index[term1.lower()] & \
           inverted_index[term2.lower()]


def search_and_not(term1, term2):
    return inverted_index[term1.lower()] - \
           inverted_index[term2.lower()]

def display_results(record_ids):
    results = df[df["record_id"].isin(record_ids)]

    if results.empty:
        print("No matching records found.")
        return

    for _, row in results.iterrows():
        print("\nRecord ID:", row["record_id"])
        print("Reference:", row["reference"])
        print("Message:", row["message"])




In [18]:

print("\nPROJECT AND NOT SUBMITTED")
display_results(search_and_not("project", "submitted"))



PROJECT AND NOT SUBMITTED

Record ID: U01
Reference: UNI/26/541
Message: An inquiry about project reached the desk. Ref: UNI/26/541.

Record ID: U10
Reference: UNI/26/550
Message: A student contacted the office regarding project paperwork. Ref: UNI/26/550.

Record ID: U11
Reference: UNI/26/551
Message: A caller mentioned project but supplied few details. Ref: UNI/26/551.

Record ID: U13
Reference: UNI/26/553
Message: The academic team opened a fresh inquiry about project. Ref: UNI/26/553.

Record ID: U15
Reference: UNI/26/555
Message: A new note concerning project was added to the record. Reference UNI/26/555.


In [19]:

print("\nMixed-case query: PrOjEcT AND SuBmItTeD")
display_results(search_and("PrOjEcT", "SuBmItTeD"))



Mixed-case query: PrOjEcT AND SuBmItTeD

Record ID: U02
Reference: UNI/26/542
Message: DO NOT ignore the project issue: submitted was recorded. Ref: UNI/26/542.

Record ID: U06
Reference: UNI/26/546
Message: The project request is submitted in the student portal. Reference UNI/26/546. Logged: 04/09/2026.

Record ID: U09
Reference: UNI/26/549
Message: PROJECT SUBMITTED! Please verify this item promptly. Reference UNI/26/549.

Record ID: U18
Reference: UNI/26/558
Message: The academic record for project was marked submitted by the office. Reference UNI/26/558. Logged: 16/09/2026.


In [20]:

print("\nQuery containing an absent term")
display_results(search_and("project", "scholarshipxyz"))


Query containing an absent term
No matching records found.


Consider record U10:
A student contacted the office regarding project paperwork.

This message may be relevant to someone searching for project submission paperwork. However, it does not contain the exact word submitted, so project AND submitted will not retrieve it.
A more advanced search system could use synonyms or semantic similarity to find messages with related meanings.

Next TASK


#**Task** B — Statistical Language Modelling (CO2)
Using Q22_Corpus_CO2.txt, develop a bigram language model that represents word transitions,
including the start and end of each sentence. Identify the likely next words following “mentors”
and their estimated probabilities. Calculate the probability of “mentors submit project updates”,
then compare the perplexities of “mentors submit schedule details” and “staff submit project
updates”. Explain the role of an unseen word transition in the results.

In [16]:

from collections import Counter
import math

# 1. Load the training sentences
with open("Q22_Corpus_CO2.txt", "r") as file:
    sentences = [
        line.strip().lower()
        for line in file
        if line.strip()
    ]

# 2. Build bigram and history counts
bigram_counts = Counter()
history_counts = Counter()

for sentence in sentences:
    words = ["<s>"] + sentence.split() + ["</s>"]

    for i in range(len(words) - 1):
        bigram_counts[(words[i], words[i + 1])] += 1
        history_counts[words[i]] += 1

# 3. Calculate bigram probability
def bigram_probability(previous, current):
    denominator = history_counts[previous]

    if denominator == 0:
        return 0

    return bigram_counts[(previous, current)] / denominator

# 4. Find next words after mentors
print("Next words after mentors:")
for (previous, current), count in bigram_counts.items():
    if previous == "mentors":
        probability = bigram_probability(previous, current)
        print(current, "Count:", count,
              "Probability:", round(probability, 4))

# 5. Calculate sentence probability
def sentence_probability(sentence):
    words = ["<s>"] + sentence.lower().split() + ["</s>"]
    probability = 1.0

    for i in range(len(words) - 1):
        probability *= bigram_probability(
            words[i], words[i + 1]
        )

    return probability

# 6. Calculate perplexity
# N counts actual words, excluding <s> and </s>
def perplexity(sentence):
    probability = sentence_probability(sentence)
    N = len(sentence.split())

    if probability == 0:
        return float("inf")

    return probability ** (-1 / N)

# 7. Required sentences
sentences_to_test = [
    "mentors submit project updates",
    "mentors submit schedule details",
    "staff submit project updates"
]

for sentence in sentences_to_test:
    print("\nSentence:", sentence)
    print("Probability:", sentence_probability(sentence))
    print("Perplexity:", perplexity(sentence))


Next words after mentors:
submit Count: 4 Probability: 0.5714
discuss Count: 3 Probability: 0.4286

Sentence: mentors submit project updates
Probability: 0.05555555555555555
Perplexity: 2.0597671439071177

Sentence: mentors submit schedule details
Probability: 0.018518518518518517
Perplexity: 2.7108060108295344

Sentence: staff submit project updates
Probability: 0.0
Perplexity: inf


An unseen transition means that two consecutive words never appear together in the training data.
For example, mentors → submit appears four times in our corpus, whereas staff → submit never occurs.
In a bigram model, an unseen transition gets a probability of zero if we do not use smoothing. Since the probability of a sentence is calculated by multiplying the probabilities of its word transitions, even one zero makes the probability of the entire sentence zero. As a result, the perplexity of staff submit project updates becomes infinite.
We can solve this problem using smoothing techniques, such as Laplace smoothing, which assign a small non-zero probability to unseen transitions. However, since the question does not specify any smoothing method, we use the basic bigram model without smoothing for our calculations.